In [0]:
import requests
import json

In [0]:
def fetch_pokemon_page(base_url, endpoint, offset, limit):
    response = requests.get(
        f"{base_url}{endpoint}",
        params={"offset": offset, "limit": limit},
        timeout=30
    )

    response.raise_for_status()

    return {
        "raw_payload": response.text,
        "parsed_payload": response.json(),
        "request_url": response.url,
        "status_code": response.status_code
    }

In [0]:
def fetch_pokemon_index(base_url, endpoint, limit=20):

    pages = []
    offset = 0

    while True:

        page = fetch_pokemon_page(
            base_url=base_url,
            endpoint=endpoint,
            offset=offset,
            limit=limit
        )

        parsed_payload = page["parsed_payload"]

        pages.append({
            "raw_payload": page["raw_payload"],
            "request_url": page["request_url"],
            "offset": offset,
            "limit": limit,
            "status_code": page["status_code"]
        })

        records = parsed_payload["results"]

        print(
            f"Offset: {offset} | "
            f"Records: {len(records)} | "
            f"Pages acquired: {len(pages)}"
        )

        if parsed_payload["next"] is None:
            break

        offset += limit

    return pages

In [0]:
def extract_pokemon_urls(pages):

    pokemon_urls = []

    for page in pages:

        payload = json.loads(page["raw_payload"])

        for record in payload["results"]:
            pokemon_urls.append({
                "name": record["name"],
                "url": record["url"]
            })

    return pokemon_urls

In [0]:
def fetch_pokemon_detail(url):

    response = requests.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    return {
        "raw_payload": response.text,
        "request_url": response.url,
        "status_code": response.status_code
    }

In [0]:
def fetch_all_pokemon_details(pokemon_urls):

    details = []

    for index, pokemon in enumerate(pokemon_urls, start=1):

        detail = fetch_pokemon_detail(
            pokemon["url"]
        )

        details.append({
            "raw_payload": detail["raw_payload"],
            "request_url": detail["request_url"],
            "status_code": detail["status_code"]
        })

        print(
            f"Fetched: {index}/{len(pokemon_urls)} | "
            f"Pokemon: {pokemon['name']}"
        )

    return details